In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [25]:
df = pd.read_csv("titanic.csv")

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [26]:
df.shape

(891, 15)

## 1. Выбор признаков и целевой переменной

In [27]:
features = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked",
    "deck"
]

X = df[features]
y = df["survived"]

X.head()

,pclass,sex,age,sibsp,parch,fare,embarked,deck
0,3,male,22.0,1,0,7.2500,S,NaN
1,1,female,38.0,1,0,71.2833,C,C
2,3,female,26.0,0,0,7.9250,S,NaN
3,1,female,35.0,1,0,53.1000,S,C
4,3,male,35.0,0,0,8.0500,S,NaN


In [28]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    891 non-null    int64  
 1   sex       891 non-null    str    
 2   age       714 non-null    float64
 3   sibsp     891 non-null    int64  
 4   parch     891 non-null    int64  
 5   fare      891 non-null    float64
 6   embarked  889 non-null    str    
 7   deck      203 non-null    str    
dtypes: float64(2), int64(3), str(3)
memory usage: 55.8 KB


In [29]:
X.isnull().sum()

pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
deck        688
dtype: int64

## 2. Разделение на обучающую и тестовую выборки

In [30]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [31]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (712, 8)
X_test: (179, 8)
y_train: (712,)
y_test: (179,)


In [32]:
numeric_features = [
    "age",
    "sibsp",
    "parch",
    "fare"
]

categorical_features = [
    "sex",
    "embarked",
    "pclass"
]

deck_feature = [
    "deck"
]

In [33]:
numeric_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    )
])

In [34]:
categorical_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

In [35]:
deck_pipeline = Pipeline(steps=[
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )
    ),
    (
        "encoder",
        OneHotEncoder(handle_unknown="ignore")
    )
])

In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        ),
        (
            "deck",
            deck_pipeline,
            deck_feature
        )
    ]
)

In [37]:
X_train_processed = preprocessor.fit_transform(X_train)

In [38]:
X_test_processed = preprocessor.transform(X_test)

In [39]:
print("Train до обработки:", X_train.shape)
print("Test до обработки:", X_test.shape)

print("Train после обработки:", X_train_processed.shape)
print("Test после обработки:", X_test_processed.shape)

Train до обработки: (712, 8)
Test до обработки: (179, 8)
Train после обработки: (712, 20)
Test после обработки: (179, 20)


In [40]:
assert (
    X_train_processed.shape[1]
    == X_test_processed.shape[1]
)

print("Размерности признаков совпадают.")
print("Утечки при обучении preprocessor нет.")

Размерности признаков совпадают.
Утечки при обучении preprocessor нет.


In [41]:
print(X_train_processed.shape)
print(X_test_processed.shape)

(712, 20)
(179, 20)


In [42]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)
print("Количество признаков:", len(feature_names))

['num__age' 'num__sibsp' 'num__parch' 'num__fare' 'cat__sex_female'
 'cat__sex_male' 'cat__embarked_C' 'cat__embarked_Q' 'cat__embarked_S'
 'cat__pclass_1' 'cat__pclass_2' 'cat__pclass_3' 'deck__deck_A'
 'deck__deck_B' 'deck__deck_C' 'deck__deck_D' 'deck__deck_E'
 'deck__deck_F' 'deck__deck_G' 'deck__deck_Unknown']
Количество признаков: 20


In [43]:
print("Пропусков в train:", np.isnan(X_train_processed).sum())
print("Пропусков в test:", np.isnan(X_test_processed).sum())

Пропусков в train: 0
Пропусков в test: 0


In [44]:
print("Train shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)

assert X_train_processed.shape[1] == X_test_processed.shape[1]

print("Размерности признаков совпадают.")
print("Pipeline обучен только на train и применён к test.")

Train shape: (712, 20)
Test shape: (179, 20)
Размерности признаков совпадают.
Pipeline обучен только на train и применён к test.


## 5. Обоснование стратегий предобработки

1. `age` — числовой признак. Пропуски заполняются медианой, потому что медиана устойчивее среднего к выбросам и перекошенному распределению.

2. `fare` — числовой признак. Используется медиана, так как в EDA был обнаружен сильный правый перекос и большое количество потенциальных выбросов.

3. `sibsp` и `parch` — числовые признаки. Пропуски при необходимости также заполняются медианой. После этого числовые признаки масштабируются с помощью StandardScaler.

4. `sex`, `embarked` и `pclass` — категориальные признаки. Пропуски заполняются наиболее частым значением, после чего применяется OneHotEncoder.

5. `deck` содержит около 77% пропущенных значений. Вместо заполнения наиболее частым значением используется отдельная категория `Unknown`, чтобы сохранить информацию о самом факте отсутствия данных.

6. `OneHotEncoder(handle_unknown="ignore")` позволяет корректно обрабатывать категории, которые могут появиться в test, но отсутствовать в train.

7. Для предотвращения утечки данных пайплайн обучается только на `X_train` через `fit_transform()`, а к `X_test` применяется только `transform()`.

## 6. Итог

После предобработки:
- обучающая выборка имеет размерность `(712, 20)`;
- тестовая выборка имеет размерность `(179, 20)`;
- количество признаков совпадает;
- пропуски полностью устранены;
- параметры заполнения, кодирования и масштабирования были вычислены только по обучающей выборке.

Таким образом, пайплайн предобработки собран без утечки информации из test в train.